# LangChain Basic & Agentic Modules


Welcome! This notebook is designed to be worked through top to bottom. It assumes you're **comfortable with Python** but **new to LangChain**, so we'll focus entirely on LangChain concepts, building up from raw LLM calls to a full agentic, graph-based application.

## Roadmap (approx. timing)

| # | Section | Time |
|---|---------|------|
| 0 | Why LangChain? What is it? What does it contain?
| 1 | Setup (HuggingFace ecosystem)
| 2 | Models — LLMs & Chat Models
| 3 | Prompts & Output Parsers
| 4 | LCEL — The Expression Language
| 5 | Memory & Conversation History
| 6 | Document Loading & Text Splitting
| 7 | Embeddings & Vector Stores
| 8 | Retrievers
| 9 | RAG — Retrieval-Augmented Generation
| 10 | Tools
| 11 | Wrap-up & Next Steps

**Prerequisites:**
- A free [HuggingFace account](https://huggingface.co/join) and [access token](https://huggingface.co/settings/tokens)

Let's start with *why this framework exists at all* — that context makes everything else click faster.

---
## 0. What We Had Before, What Was Lacking, and Why LangChain

### What we had before (calling an LLM "raw")

Before frameworks like LangChain, using an LLM in an application usually meant calling a provider's SDK or REST API directly, and hand-rolling everything else yourself: prompt strings, conversation history, retries, parsing the output, etc.

Let's see what that actually looks like, using the HuggingFace `transformers` library directly (no LangChain).

In [ ]:
# NOTE: This cell is illustrative — you don't need to run it if you don't have
# a local model downloaded. It exists to show the pain points we're about to solve.

# from transformers import pipeline
#
# generator = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
#
# def ask(question, history_text=""):
#     # We have to hand-build the prompt string ourselves, every time
#     prompt = f"{history_text}\nUser: {question}\nAssistant:"
#     output = generator(prompt, max_new_tokens=100)[0]["generated_text"]
#     return output
#
# # Turn 1
# answer1 = ask("What's the capital of France?")
# print(answer1)
#
# # Turn 2 — the model has NO memory of turn 1 unless we manually
# # concatenate the entire conversation ourselves, every single call.
# answer2 = ask("What's its population?", history_text=f"User: What's the capital of France?\nAssistant: {answer1}")
# print(answer2)

print("See the markdown above — this cell is for reading, not running.")

### What was lacking

Doing it "raw" like above works for a demo, but breaks down fast in a real application:

- **No standard interface** — every provider (OpenAI, Anthropic, HuggingFace, local models) has a different SDK, different request/response shape, different auth. Swapping models means rewriting code.
- **No composability** — there's no clean way to chain "retrieve context → build prompt → call model → parse output → call another model" without writing custom glue code every time.
- **No memory abstraction** — conversation history has to be manually tracked, formatted, and truncated by hand.
- **No retrieval/grounding** — connecting an LLM to *your* documents (RAG) means writing your own chunking, embedding, vector search, and prompt-stuffing logic from scratch.
- **No tool use / agency** — letting a model call functions, search the web, query a database, and reason about *when* to do so requires a hand-built loop.
- **No standard output parsing** — getting reliable structured data (JSON, specific fields) out of free-text generation is finicky and provider-specific.
- **No orchestration for multi-step workflows** — anything beyond "one prompt in, one answer out" (branching, loops, retries, human approval steps) needs custom control-flow code.

### Why LangChain

LangChain exists to turn all of the above into **reusable, standardized, composable building blocks**, so you write your *application logic*, not your *plumbing*.

### What LangChain *is*

> LangChain is an open-source framework for building applications powered by LLMs. It treats every step of an LLM workflow — prompting, calling a model, parsing output, retrieving documents, calling tools — as a **Runnable**: a standard, swappable, composable unit that can be chained together.

### What we *do* in LangChain

Typical things people build with it:
- Chatbots with memory
- RAG systems (chat with your own documents)
- Agents that can use tools (search, calculators, APIs, databases)
- Structured data extraction pipelines
- Multi-step, stateful workflows (via LangGraph) — including ones with loops and human-in-the-loop approval

### What LangChain *contains* (the ecosystem map)

```
┌─────────────────────────────────────────────────────────────────┐
│                         Your Application                        │
├─────────────────────────────────────────────────────────────────┤
│  LangSmith        — observability, tracing, evaluation (SaaS)   │
│  LangServe        — deploy chains as a REST API                 │
│  LangGraph        — stateful, graph-based agent orchestration   │
├─────────────────────────────────────────────────────────────────┤
│  langchain          — chains, agents, retrieval strategies      │
│  langchain-community— 3rd-party integrations (loaders, vector   │
│                        stores, tools contributed by the         │
│                        community)                                │
│  langchain-huggingface / langchain-openai / langchain-anthropic  │
│                      — official "partner" packages per provider  │
├─────────────────────────────────────────────────────────────────┤
│  langchain-core    — the foundation: Runnables, LCEL, prompts,  │
│                       output parsers, base interfaces            │
└─────────────────────────────────────────────────────────────────┘
```

We're going to build our way up this stack, from the bottom (`langchain-core`) to the top (`LangGraph`), using **HuggingFace** as our model provider throughout.

---
## 1. Setup

We'll use the `langchain-huggingface` partner package. It gives us two ways to run a model:

1. **`HuggingFaceEndpoint`** — calls a model hosted on HuggingFace's Inference API (or your own Inference Endpoint). Fast to set up, no local compute needed. **We'll use this as our primary path.**
2. **`HuggingFacePipeline`** — runs a model locally via `transformers`. No API token needed, but requires local compute/RAM and a model download. We'll show it as a fallback.

Install what we need:

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface \
    langchain-chroma langchain-text-splitters langgraph langchainhub \
    sentence-transformers chromadb huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6

### Get a HuggingFace access token

1. Create a free account at https://huggingface.co/join
2. Go to https://huggingface.co/settings/tokens and create a token (a "Read" token is enough for inference)
3. Run the cell below and paste it in when prompted (it won't be echoed to the notebook)

> **Note on model availability:** HuggingFace's free serverless Inference API only hosts a rotating subset of models at any time. If a `repo_id` below gives you a "model not supported" error, check https://huggingface.co/models?inference=warm for a currently available chat model and swap the `repo_id`.

In [ ]:
import os
from getpass import getpass

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass("Enter your HuggingFace access token: ")

print("Token set:", "HUGGINGFACEHUB_API_TOKEN" in os.environ)

Enter your HuggingFace access token: ··········
Token set: True


### Finding Models Supported by Inference Providers

Hugging Face routes model calls through third-party **Inference Providers** (Together AI, Fireworks, Novita, Cerebras, etc.) rather than hosting every model itself. This means not every model on the Hub can actually be called via the API — only ones that a provider has chosen to host.

This snippet queries the Hub for text-generation models that currently have at least one active provider behind them, and prints the first 20 `repo_id`s.

The purpose is to avoid guessing a model name and hitting a `model_not_supported` error. Instead of assuming an older or less common model (like `zephyr-7b-alpha`) is still callable, this pulls a live, filtered list of models that are actually usable right now through `router.huggingface.co`.

One thing to keep in mind: this only confirms that *some* provider hosts the model — it doesn't check whether *you* have that specific provider enabled on your account (`huggingface.co/settings/inference-providers`). A model showing up here can still fail if the provider serving it isn't turned on for your token.

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token="hf_xxx")
models = api.list_models(inference_provider="all", pipeline_tag="text-generation")
for m in list(models)[:20]:
    print(m.id)

deepseek-ai/DeepSeek-V4-Flash-0731
zai-org/GLM-5.2
deepseek-ai/DeepSeek-V4-Flash
poolside/Laguna-S-2.1
prism-ml/Ternary-Bonsai-27B-gguf
deepseek-ai/DeepSeek-V4-Pro
meta-llama/Llama-3.1-8B-Instruct
Qwen/Qwen2.5-7B-Instruct
Qwen/Qwen3-8B
Qwen/Qwen3-0.6B
openai/gpt-oss-120b
deepseek-ai/DeepSeek-R1
openai/gpt-oss-20b
tencent/Hy3
XiaomiMiMo/MiMo-V2.5
empero-ai/Qwythos-9B-Claude-Mythos-5-1M
deepseek-ai/DeepSeek-V3
nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4
Qwen/Qwen3-Coder-Next
deepreinforce-ai/Ornith-1.0-9B


---
## 2. Models — LLMs & Chat Models

LangChain distinguishes between two model interfaces:

- **LLM** — text in, text in → `.invoke("some string")` returns a string. Older style, still used for base (non-chat-tuned) models.
- **Chat Model** — a list of *messages* in (system/human/ai) → an `AIMessage` out. This is the standard for modern instruction-tuned models, and what you'll use almost all the time.

`ChatHuggingFace` wraps a `HuggingFaceEndpoint` (or `HuggingFacePipeline`) and applies the correct chat template for the underlying model, so you get a proper chat-model interface even though the endpoint itself is just a text-generation API.

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# Using a smaller model (Gemma-3-1B) to stay within free usage limits
llm_endpoint = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Flash",
    task="text-generation",
    max_new_tokens=256,
    temperature=0.7,
)

# Wrap it so it behaves like a proper chat model
chat_model = ChatHuggingFace(llm=llm_endpoint)

response = chat_model.invoke("In one sentence, what is LangChain?")
print(type(response))
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
LangChain is an open-source framework designed to simplify the development of applications powered by large language models (LLMs) by providing modular components for chaining prompts, managing data connections, and integrating with external tools.


### Streaming, batching, and message lists

Chat models support more than a single `.invoke()`:

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Streaming — get tokens as they're generated instead of waiting for the full response
print("--- Streaming ---")
for chunk in chat_model.stream("Name three uses of a vector database, briefly."):
    print(chunk.content, end="", flush=True)
print()

# Batching — send multiple independent prompts in one call
print("\n--- Batch ---")
results = chat_model.batch(["What is a token in NLP?", "What is an embedding?"])
for r in results:
    print("-", r.content[:80], "...")

# Message lists — give explicit roles (system / human)
print("\n--- With a system message ---")
messages = [
    SystemMessage(content="You are a terse assistant. Answer in <=10 words."),
    HumanMessage(content="What does RAG stand for?"),
]
print(chat_model.invoke(messages).content)

--- Streaming ---
Here are three uses of a vector database:

1.  **Semantic Search:** Enables searching by meaning rather than exact keywords (e.g., finding "funny cat videos" even if the caption says "humorous feline clips").
2.  **Recommendation Systems:** Powers "similar items" features by finding products, movies, or articles with the closest vector embeddings to a user's past preferences.
3.  **Retrieval-Augmented Generation (RAG):** Provides large language models (LLMs) with relevant, up-to-date context by quickly retrieving the most pertinent documents or data chunks before generating an answer.

--- Batch ---
- This is a fundamental concept in Natural Language Processing (NLP).

In simple t ...
- This is a fundamental concept in modern machine learning and AI. Here’s a compre ...

--- With a system message ---
Retrieval-Augmented Generation


### Fallback: running fully local (no API token)

If you'd rather not use the Inference API, `HuggingFacePipeline` runs a small model locally via `transformers`. It's slower on CPU but needs no token and no internet at inference time.

```python
from langchain_huggingface import HuggingFacePipeline

local_llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 200},
)
local_chat_model = ChatHuggingFace(llm=local_llm)
print(local_chat_model.invoke("Hello!").content)
```

We'll keep using `chat_model` (the Inference API version) for the rest of the notebook — just know you can substitute `local_chat_model` anywhere.

---
## 3. Prompts & Output Parsers

### Prompt templates

Hard-coding prompt strings with f-strings works until you need to reuse, version, or compose them. `ChatPromptTemplate` gives you a reusable, parameterized template with defined roles.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant for {company} employees. Be concise."),
    ("human", "{question}"),
])

# .invoke() fills in the variables and returns a formatted list of messages
formatted = prompt.invoke({"company": "Acme Corp", "question": "How do I reset my password?"})
print(formatted.to_messages())

[SystemMessage(content='You are a helpful assistant for Acme Corp employees. Be concise.', additional_kwargs={}, response_metadata={}), HumanMessage(content='How do I reset my password?', additional_kwargs={}, response_metadata={})]


### Output parsers

A chat model always returns an `AIMessage` object. Most of the time you just want the text — or better yet, *structured* data. Output parsers handle that conversion.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response = chat_model.invoke("Say hello in French.")
print("Raw AIMessage:", raw_response)
print("Parsed string:", parser.invoke(raw_response))

Raw AIMessage: content='Bonjour !' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 9, 'total_tokens': 13}, 'model_name': 'deepseek-ai/DeepSeek-V4-Flash', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fcc65-07e4-7301-8c72-2add921a779f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 9, 'output_tokens': 4, 'total_tokens': 13}
Parsed string: Bonjour !


### Structured output with Pydantic

For real applications you often want *specific fields*, not free text. `PydanticOutputParser` (or a model's native `.with_structured_output()`) lets you define a schema and get validated Python objects back.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define the schema
class ContactInfo(BaseModel):
    name: str = Field(description="The person's full name")
    department: str = Field(description="Department the person works in, e.g. HR or IT")

# Turns the defined schema into instructions the model can follow
struct_parser = PydanticOutputParser(pydantic_object=ContactInfo)

# A structured boilerplate that langchain insert everytime before generation
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the requested fields.\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=struct_parser.get_format_instructions())

# We'll actually run this end-to-end once we cover LCEL in the next section —
# for now, just look at the format instructions the parser generates for the model:
print(struct_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "The person's full name", "title": "Name", "type": "string"}, "department": {"description": "Department the person works in, e.g. HR or IT", "title": "Department", "type": "string"}}, "required": ["name", "department"]}
```


---
## 4. LCEL — The LangChain Expression Language

This is the core idea that makes LangChain *composable*: every building block (prompt, model, parser, retriever, tool...) implements the same **`Runnable`** interface — `.invoke()`, `.stream()`, `.batch()`, and their async equivalents (`.ainvoke()`, etc.).

Because they all share this interface, you can **pipe them together with `|`**, the same way you'd pipe shell commands.

In [ ]:
# prompt | model | parser  — this is THE most common LangChain pattern you'll write
extraction_chain = structured_prompt | chat_model | struct_parser

result = extraction_chain.invoke({"text": "Hi, I'm Aamir Khan Maarofi and I work in the FCSE GIKI."})
print(result)
print(type(result), "-> name:", result.name, "| department:", result.department)

name='Aamir Khan Maarofi' department='FCSE'
<class '__main__.ContactInfo'> -> name: Aamir Khan Maarofi | department: FCSE


### RunnableParallel — run multiple things at once

Sometimes you want to run several chains on the same input and combine their outputs.

In [ ]:
from langchain_core.runnables import RunnableParallel

summarize_prompt = ChatPromptTemplate.from_template("Summarize in one sentence: {text}")
sentiment_prompt = ChatPromptTemplate.from_template("Classify the sentiment (positive/neutral/negative) of: {text}")

parallel_chain = RunnableParallel(
    summary=summarize_prompt | chat_model | StrOutputParser(),
    sentiment=sentiment_prompt | chat_model | StrOutputParser(),
)

result = parallel_chain.invoke({"text": "The new laptop rollout went smoothly and the team is thrilled with the upgrade."})
print(result)

{'summary': 'The new laptop rollout was seamless, and the team is delighted with the upgrade.', 'sentiment': 'positive'}


### RunnablePassthrough — forward input unchanged alongside other data

Very common in RAG: you need both the *retrieved context* AND the *original question* to reach the final prompt.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def word_count(inputs: dict) -> dict:
    inputs["word_count"] = len(inputs["text"].split())
    return inputs

annotate_chain =  RunnablePassthrough() | RunnableLambda(word_count)

print(annotate_chain.invoke({"text": "LangChain makes LLM pipelines composable"}))

{'text': 'LangChain makes LLM pipelines composable', 'word_count': 5}


### RunnableBranch — conditional routing

Route to a different chain depending on the input, without writing an `if/else` around your `.invoke()` calls.

In [ ]:
from langchain_core.runnables import RunnableBranch

it_prompt = ChatPromptTemplate.from_template("Answer this IT support question concisely: {question}")
hr_prompt = ChatPromptTemplate.from_template("Answer this HR policy question concisely: {question}")
general_prompt = ChatPromptTemplate.from_template("Answer this question concisely: {question}")

router = RunnableBranch(
    (lambda x: "password" in x["question"].lower() or "vpn" in x["question"].lower(), it_prompt | chat_model | StrOutputParser()),
    (lambda x: "leave" in x["question"].lower() or "payroll" in x["question"].lower(), hr_prompt | chat_model | StrOutputParser()),
    general_prompt | chat_model | StrOutputParser(),   # default branch
)

print(router.invoke({"question": "How do I reset my VPN password?"}))

To reset your VPN password:

1. **Contact your IT admin** – Most corporate VPNs require admin reset.
2. **If self-service** – Go to your company’s password reset portal (e.g., Okta, Azure AD) and follow the prompts.
3. **VPN client** – Open the VPN app, look for “Forgot Password” or “Reset Password” on the login screen.
4. **Check email** – A reset link may be sent to your work email.

If unsure, ask your IT help desk for the specific process.


---
## 5. Memory & Conversation History

Chat models are **stateless** — every `.invoke()` call is independent. To build a chatbot that "remembers" earlier turns, LangChain wraps a chain with `RunnableWithMessageHistory`, backed by a `ChatMessageHistory` store keyed by a `session_id`.

(Older `ConversationBufferMemory`-style classes still exist but are considered legacy — this is the current recommended pattern.)

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{history}"),
    ("human", "{input}"),
])

base_chain = chat_prompt | chat_model | StrOutputParser()

# A simple in-memory store: {session_id: ChatMessageHistory}
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user-123"}}

turn1 = chain_with_memory.invoke({"input": "My name is Sam and I work in IT."}, config=config)
print("Turn 1:", turn1)

turn2 = chain_with_memory.invoke({"input": "What department do I work in?"}, config=config)
print("Turn 2:", turn2)   # the model should recall "IT" from turn 1

# A different session_id has completely separate memory
turn3 = chain_with_memory.invoke({"input": "What department do I work in?"}, config={"configurable": {"session_id": "user-456"}})
print("Turn 3 (new session):", turn3)   # should NOT know — no memory of Sam

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Turn 1: Hello Sam! Nice to meet you. How can I assist you with IT-related questions or tasks today?
Turn 2: Based on what you've told me, you work in **IT**. However, if you're asking which specific department within IT (like Network Security, Help Desk, Software Development, etc.), you haven't mentioned that yet.
Turn 3 (new session): I don't have access to personal information about you, including where you work or which department you're in. If you'd like to share that context, I’d be happy to help with anything related to your role or tasks!


In [ ]:
for session_id, history in store.items():
    print(f"=== Session: {session_id} ===")
    for msg in history.messages:
        role = msg.type  # "human" or "ai"
        print(f"  [{role}] {msg.content}")
    print()

=== Session: user-123 ===
  [human] My name is Sam and I work in IT.
  [ai] Hello Sam! Nice to meet you. How can I assist you with IT-related questions or tasks today?
  [human] What department do I work in?
  [ai] Based on what you've told me, you work in **IT**. However, if you're asking which specific department within IT (like Network Security, Help Desk, Software Development, etc.), you haven't mentioned that yet.

=== Session: user-456 ===
  [human] What department do I work in?
  [ai] I don't have access to personal information about you, including where you work or which department you're in. If you'd like to share that context, I’d be happy to help with anything related to your role or tasks!



---
## 6. Document Loading & Text Splitting

This is where we start building toward **RAG**. We need documents to retrieve from — let's create a small synthetic knowledge base (HR + IT policy docs) that we'll reuse for the rest of the notebook.

### Why splitting matters
- Models have a limited context window — you can't stuff an entire wiki into one prompt.
- Retrieval works best over focused chunks, not giant documents — a 5,000-word doc mostly won't be relevant to a specific question.
- `RecursiveCharacterTextSplitter` tries to split on natural boundaries (paragraphs → sentences → words) so chunks stay coherent, with a small `chunk_overlap` so context isn't lost at chunk boundaries.

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.makedirs("knowledge_base", exist_ok=True)

hr_policy = """HR Policy Handbook

Password Policy (HR Systems): Employees must reset their HR portal password every 90 days.
Contact hr-support@acme.com for portal password resets. This does NOT apply to IT systems.

Leave Policy: Full-time employees accrue 1.5 days of paid leave per month. Leave requests
must be submitted at least 5 business days in advance through the HR portal.

Payroll: Salaries are disbursed on the last working day of each month. Payroll queries
should be directed to payroll@acme.com.

Onboarding: New employees complete HR orientation on their first day, followed by
department-specific onboarding with their manager."""

it_policy = """IT Policy Handbook

Password Policy (IT Systems): Employees must reset their network/VPN password every 60 days
via the IT self-service portal. Passwords must be at least 12 characters. Contact
it-helpdesk@acme.com for VPN or network password resets. This does NOT apply to the HR portal.

VPN Access: VPN access is granted automatically on your first day and must be renewed
annually through the IT portal.

Hardware Requests: Laptop and peripheral requests go through the IT ticketing system,
with a standard turnaround of 3-5 business days.

Software Installation: Only IT-approved software may be installed on company devices.
Requests for new software require manager approval."""

with open("knowledge_base/hr_policy.txt", "w") as f:
    f.write(hr_policy)
with open("knowledge_base/it_policy.txt", "w") as f:
    f.write(it_policy)

# Load
hr_docs = TextLoader("knowledge_base/hr_policy.txt").load()
it_docs = TextLoader("knowledge_base/it_policy.txt").load()

# Tag each with department metadata BEFORE splitting so every chunk inherits it
for d in hr_docs:
    d.metadata["department"] = "HR"
for d in it_docs:
    d.metadata["department"] = "IT"

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
hr_chunks = splitter.split_documents(hr_docs)
it_chunks = splitter.split_documents(it_docs)

all_chunks = hr_chunks + it_chunks
print(f"HR chunks: {len(hr_chunks)} | IT chunks: {len(it_chunks)} | Total: {len(all_chunks)}")
print("\nExample chunk:\n", all_chunks[0].page_content, "\nMetadata:", all_chunks[0].metadata)

HR chunks: 4 | IT chunks: 6 | Total: 10

Example chunk:
 HR Policy Handbook

Password Policy (HR Systems): Employees must reset their HR portal password every 90 days.
Contact hr-support@acme.com for portal password resets. This does NOT apply to IT systems. 
Metadata: {'source': 'knowledge_base/hr_policy.txt', 'department': 'HR'}


---
## 7. Embeddings & Vector Stores

- **Embeddings** turn text into a vector of numbers such that semantically similar text ends up close together in vector space.
- A **vector store** indexes those vectors so you can efficiently find "the chunks most similar to this query" — that's the core operation behind retrieval.

We'll use `sentence-transformers/all-MiniLM-L6-v2` via `HuggingFaceEmbeddings` — it runs **locally on CPU**, is fast, and needs no API calls (unlike our chat model, which uses the hosted Inference API).

For the vector store we'll use **Chroma** — a lightweight, local, open-source vector database. No account or API key required.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Quick sanity check on the embedding itself
vec = embeddings.embed_query("password reset")
print("Embedding dimension:", len(vec))

# Build the vector store from our chunks
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="acme_policies",
)

# Plain similarity search
results = vectorstore.similarity_search("How often do I need to change my password?", k=3)
for r in results:
    print(f"[{r.metadata['department']}]", r.page_content[:80], "...")

# With similarity scores (lower = more similar, for Chroma's default distance metric)
print("\n--- with scores ---")
scored = vectorstore.similarity_search_with_score("How often do I need to change my password?", k=2)
for doc, score in scored:
    print(f"score={score:.4f} [{doc.metadata['department']}]", doc.page_content[:60])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
[HR] HR Policy Handbook

Password Policy (HR Systems): Employees must reset their HR  ...
[IT] Password Policy (IT Systems): Employees must reset their network/VPN password ev ...
[IT] it-helpdesk@acme.com for VPN or network password resets. This does NOT apply to  ...

--- with scores ---
score=0.9288 [HR] HR Policy Handbook

Password Policy (HR Systems): Employees 
score=0.9495 [IT] Password Policy (IT Systems): Employees must reset their net


---
## 8. Retrievers

A **retriever** is a `Runnable` wrapper around a vector store's search — this is what makes it composable with everything else via LCEL (`|`).

`vectorstore.as_retriever()` gives you one, with `search_kwargs` to control behavior:
- `k` — how many chunks to return
- `filter` — restrict search to documents matching metadata (this is the fix for the department-scoping bug from earlier — a **hard filter at the vector-store level**, not a soft similarity re-rank)
- `search_type="mmr"` — Maximal Marginal Relevance, which balances relevance with *diversity* among results (useful when top-k results are near-duplicates)

Let's rebuild the HR-only / IT-only scoped retrievers properly this time.

In [ ]:
hr_retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"department": "HR"}})
it_retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"department": "IT"}})

hr_hits = hr_retriever.invoke("password")
it_hits = it_retriever.invoke("password")

print("HR-scoped retriever, departments returned:", {d.metadata["department"] for d in hr_hits})
print("IT-scoped retriever, departments returned:", {d.metadata["department"] for d in it_hits})
# Both sets should now contain ONLY their own department — true isolation,
# enforced by Chroma's metadata filter at query time, not by chance.

HR-scoped retriever, departments returned: {'HR'}
IT-scoped retriever, departments returned: {'IT'}


**Other retriever variants worth knowing (not run here, just so you recognize the names):**
- `MultiQueryRetriever` — asks the LLM to generate several rephrasings of the query, retrieves for each, and de-duplicates. Helps when a single phrasing misses relevant chunks.
- `ContextualCompressionRetriever` — retrieves broadly, then uses an LLM or embedding filter to trim each chunk down to only the relevant sentences.
- `EnsembleRetriever` — combines a dense retriever (embeddings) with a sparse one (BM25/keyword) and merges rankings — "hybrid search."
- `SelfQueryRetriever` — lets the LLM translate a natural-language question (*"IT policies about VPN from last year"*) into a structured metadata filter automatically.

---
## 9. RAG — Retrieval-Augmented Generation

Now we connect retrieval to generation: retrieve relevant chunks, stuff them into the prompt as context, and have the model answer **grounded in that context** instead of its own (possibly outdated or hallucinated) knowledge.

### Build it manually with LCEL first, so you see exactly what's happening

In [ ]:
from langchain_core.documents import Document

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(f"[{d.metadata.get('department','?')}] {d.page_content}" for d in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY using the provided context. If the answer isn't in the context, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

answer = rag_chain.invoke("How often do IT passwords need to be reset, and who do I contact?")
print(answer)

According to the context, IT passwords (network/VPN) must be reset every 60 days, and you should contact it-helpdesk@acme.com for VPN or network password resets.


### The higher-level API: `create_retrieval_chain`

LangChain also ships a higher-level helper that does the same thing with less boilerplate, and — importantly — **returns the source documents alongside the answer**, so you can cite them.

In [ ]:
!pip uninstall langchain

Found existing installation: langchain 1.3.13
Uninstalling langchain-1.3.13:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/langchain-1.3.13.dist-info/*
    /usr/local/lib/python3.12/dist-packages/langchain/*
Proceed (Y/n)? y
  Successfully uninstalled langchain-1.3.13


In [ ]:
!pip install -q "langchain<1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 41.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.2
    Uninstalling langchain-text-splitters-1.1.2:
      Successfully uninstalled langchain-text-splitters-1.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-cl

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below.\n\n{context}"),
    ("human", "{input}"),
])

combine_docs_chain = create_stuff_documents_chain(chat_model, qa_prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

result = retrieval_chain.invoke({"input": "What's the leave policy?"})
print("Answer:", result["answer"])
print("\nSources used:")
for doc in result["context"]:
    print(" -", f"[{doc.metadata['department']}]", doc.page_content[:60], "...")

Answer: Based on the context provided, the leave policy states that full-time employees accrue 1.5 days of paid leave per month, and leave requests must be submitted at least 5 business days in advance through the HR portal.

Sources used:
 - [HR] Leave Policy: Full-time employees accrue 1.5 days of paid le ...
 - [IT] IT Policy Handbook ...
 - [IT] Password Policy (IT Systems): Employees must reset their net ...
 - [HR] Onboarding: New employees complete HR orientation on their f ...


### Conversational RAG — combining Section 5 (memory) with retrieval

Real chat-with-your-docs apps need the retriever to understand *follow-up* questions like "what about the other one?" — which requires the conversation history. LangChain's `create_history_aware_retriever` rewrites the query using chat history before retrieving.

In [ ]:
from langchain.chains import create_history_aware_retriever

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and the latest question, rephrase it as a standalone question. Do not answer it."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(chat_model, retriever, contextualize_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, combine_docs_chain)

conversational_rag_with_memory = RunnableWithMessageHistory(
    conversational_rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

cfg = {"configurable": {"session_id": "rag-demo"}}
r1 = conversational_rag_with_memory.invoke({"input": "What's the IT password policy?"}, config=cfg)
print("Q1:", r1["answer"])

r2 = conversational_rag_with_memory.invoke({"input": "And what about for HR?"}, config=cfg)
print("Q2 (follow-up):", r2["answer"])  # should correctly understand "the same, but for HR"

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Q1: Based on the context provided, the IT password policy states: Employees must reset their network/VPN password every 60 days via the IT self-service portal. Passwords must be at least 12 characters.
Q2 (follow-up): According to the context provided, for HR systems (the HR portal), employees must reset their password every 90 days. To reset it, they should contact hr-support@acme.com. This policy does NOT apply to IT systems.


---
## 10. Tools

A **tool** is just a function the LLM can choose to call — with a name, description, and typed arguments the model uses to decide *when* and *how* to call it. The `@tool` decorator turns any Python function into one.

In [ ]:
from langchain_core.tools import tool
import datetime

@tool
def get_current_time() -> str:
    """Return the current date and time."""
    return datetime.datetime.now().isoformat()

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '12 * 4 + 1'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

@tool
def search_company_policy(query: str) -> str:
    """Search Acme Corp's HR and IT policy documents for an answer."""
    docs = retriever.invoke(query)
    return format_docs(docs)

tools = [get_current_time, calculator, search_company_policy]

for t in tools:
    print(f"{t.name}: {t.description}")

# Tools are Runnables too — you can call them directly
print("\nDirect call:", calculator.invoke({"expression": "15 * 3"}))

---
## 13. Wrap-up & Next Steps

### What you built today
- Went from a raw, stateless `transformers.pipeline()` call → a fully composable LCEL chain → a memory-aware chatbot → a grounded RAG system → a tool-using agent → an explicit, stateful LangGraph.
- Learned the layered ecosystem: `langchain-core` (Runnables/LCEL) → `langchain` (chains/agents) → `langchain-community` / `langchain-huggingface` (integrations) → `LangGraph` (orchestration).
- Fixed the exact HR/IT retrieval-isolation problem from earlier — now backed by a real metadata filter at the vector-store level, not an assumption.

### Where to go from here
- **LangSmith** — free tier observability/tracing for every chain and agent run; invaluable once things get complex. https://smith.langchain.com
- **LangServe** — turn any chain into a deployable REST API in a few lines.
- **Hybrid search** — combine `Chroma`/dense retrieval with `BM25Retriever` via `EnsembleRetriever` for better recall on keyword-heavy queries.
- **Evaluation** — RAGAS or LangSmith evaluators to measure faithfulness, context precision/recall, and answer relevancy instead of eyeballing outputs.
- **Multi-agent systems** — supervisor/worker patterns, where one LangGraph agent routes tasks to specialized sub-agents.
- **Official docs** — https://python.langchain.com and https://langchain-ai.github.io/langgraph/

### A few exercises to solidify this (do these on your own)
1. Add a third department (e.g. "Finance") end-to-end: new doc → chunk → embed → scoped retriever → new tool → agent picks it up automatically.
2. Swap `search_type="mmr"` into the retriever and compare results against plain similarity search on a query with near-duplicate chunks.
3. Add a `human_review` node to the manual LangGraph before the `tools` node that requires typed confirmation before a tool actually runs (a minimal human-in-the-loop pattern).
4. Replace `StrOutputParser` in the RAG chain with a `PydanticOutputParser` that returns `{answer: str, department: str, confidence: Literal["low","medium","high"]}`.